# NLTK Complete Tutor + Experimentation Case Study
## Classical NLP from Raw Text to Linguistic Structure, Statistical Models, and Evaluation

This notebook is designed as a **teaching notebook and experimentation lab**, not merely an NLTK syntax reference.

The central objective is to understand:

1. **What each NLP concept means**
2. **Why the technique exists**
3. **How NLTK implements it**
4. **Which parameters/rules control it**
5. **How changing those settings changes results**
6. **Typical failure modes**
7. **How the method connects to modern NLP**

---

# Case Study

We analyse fictional banking and customer-support text containing:

- card payments,
- account-access problems,
- fraud reports,
- people,
- locations,
- banking terminology,
- negation,
- repeated phrases,
- positive/negative sentiment,
- ambiguous words such as **bank**.

The same corpus is reused throughout the notebook so that the transformation of text can be followed end-to-end.

---

# Topics Covered

## Text preprocessing
- Sentence tokenization
- Word tokenization
- Regex tokenization
- Treebank tokenization
- Tweet tokenization
- Multi-word expressions
- Stopword removal
- Stemming
- Lemmatization
- POS-aware lemmatization

## Linguistic annotation
- POS tagging
- Custom POS tagging
- Chunking / shallow parsing
- Named Entity Recognition

## Parsing
- Context-Free Grammars
- Constituency parsing
- Parse-tree inspection
- Probabilistic CFG
- Viterbi parsing
- Dependency representations

## Lexical semantics
- WordNet
- Synsets
- Synonyms / antonyms
- Hypernyms / hyponyms
- Meronyms / holonyms
- Semantic similarity
- Word Sense Disambiguation

## Statistical NLP
- Frequency distributions
- Conditional frequency distributions
- N-grams
- Collocations
- PMI
- Chi-square
- Likelihood ratio
- Dice coefficient
- N-gram language models
- Smoothing
- Text generation
- Perplexity

## Corpus linguistics
- Concordance
- Similar contexts
- Common contexts
- Lexical diversity
- Brown corpus
- Reuters corpus
- Gutenberg corpus
- Treebank corpus

## Classical machine learning
- Naive Bayes classification
- Feature engineering
- Error analysis

## Classical sequence tagging
- Default tagger
- Unigram tagger
- Bigram tagger
- Trigram tagger
- Backoff

## Other NLTK capabilities
- VADER sentiment
- Edit distance
- BLEU
- METEOR
- Classification metrics
- Confusion matrix

---

# Learning Philosophy

Think of classical NLP as progressively adding structure:

```text
RAW TEXT
   ↓
SENTENCES
   ↓
TOKENS
   ↓
NORMALIZED TOKENS
   ↓
POS TAGS
   ↓
PHRASES / CHUNKS
   ↓
ENTITIES
   ↓
PARSE TREES
   ↓
LEXICAL SEMANTICS
   ↓
STATISTICAL REPRESENTATIONS
   ↓
LANGUAGE MODELS / CLASSIFIERS
   ↓
EVALUATION
```

Each stage introduces assumptions.

A strong NLP practitioner asks:

> What information did this transformation preserve, and what information did it destroy?

# Part I — Environment Setup

In [1]:
import nltk

resources = [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker",
    "maxent_ne_chunker_tab",
    "words",
    "vader_lexicon",
    "brown",
    "reuters",
    "gutenberg",
    "treebank",
    "movie_reviews",
]

for resource in resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception as exc:
        print(f"{resource}: {exc}")

print("NLTK version:", nltk.__version__)

NLTK version: 3.9.2


In [2]:
from collections import Counter
from pprint import pprint

from nltk import (
    FreqDist,
    ConditionalFreqDist,
    Text,
    word_tokenize,
    sent_tokenize,
    pos_tag,
    ne_chunk,
    bigrams,
    trigrams,
    ngrams,
)

from nltk.corpus import (
    stopwords,
    wordnet as wn,
    brown,
    reuters,
    gutenberg,
    treebank,
)

from nltk.stem import (
    PorterStemmer,
    LancasterStemmer,
    SnowballStemmer,
    WordNetLemmatizer,
)

from nltk.tokenize import (
    RegexpTokenizer,
    TreebankWordTokenizer,
    TweetTokenizer,
    MWETokenizer,
)

from nltk.chunk import RegexpParser
from nltk.grammar import CFG, PCFG
from nltk.parse import ChartParser, ViterbiParser, DependencyGraph
from nltk.wsd import lesk

from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

from nltk.sentiment import SentimentIntensityAnalyzer

from nltk.lm import (
    MLE,
    Laplace,
    Lidstone,
    WittenBellInterpolated,
    KneserNeyInterpolated,
)

from nltk.lm.preprocessing import padded_everygram_pipeline

# Part II — Case Study Dataset

In [3]:
documents = [
    {
        "id": 1,
        "label": "payment",
        "text": "Arjun paid SGD 120 at Marina Bay yesterday, but the card payment was declined twice."
    },
    {
        "id": 2,
        "label": "account",
        "text": "Priya cannot access her savings account after resetting the password."
    },
    {
        "id": 3,
        "label": "fraud",
        "text": "Daniel noticed an unauthorized transaction from London and wants the card blocked immediately."
    },
    {
        "id": 4,
        "label": "payment",
        "text": "The mobile payment worked after I restarted the app. Excellent support!"
    },
    {
        "id": 5,
        "label": "account",
        "text": "My bank account is locked and login verification keeps failing."
    },
    {
        "id": 6,
        "label": "fraud",
        "text": "Someone used my credit card online. I did not make this purchase."
    },
    {
        "id": 7,
        "label": "payment",
        "text": "The transfer to Singapore was completed successfully, but the merchant has not received it."
    },
    {
        "id": 8,
        "label": "account",
        "text": "Password reset was quick and the account is accessible again."
    },
    {
        "id": 9,
        "label": "fraud",
        "text": "A suspicious card transaction appeared on my statement this morning."
    },
    {
        "id": 10,
        "label": "fraud",
        "text": "The customer reported fraudulent activity after using the card in New York."
    },
    {
        "id": 11,
        "label": "payment",
        "text": "The card payment at the restaurant went through, but the receipt was not generated."
    },
    {
        "id": 12,
        "label": "account",
        "text": "The customer changed the password and regained access to the online banking account."
    },
]

raw_text = " ".join(doc["text"] for doc in documents)

print(raw_text)

Arjun paid SGD 120 at Marina Bay yesterday, but the card payment was declined twice. Priya cannot access her savings account after resetting the password. Daniel noticed an unauthorized transaction from London and wants the card blocked immediately. The mobile payment worked after I restarted the app. Excellent support! My bank account is locked and login verification keeps failing. Someone used my credit card online. I did not make this purchase. The transfer to Singapore was completed successfully, but the merchant has not received it. Password reset was quick and the account is accessible again. A suspicious card transaction appeared on my statement this morning. The customer reported fraudulent activity after using the card in New York. The card payment at the restaurant went through, but the receipt was not generated. The customer changed the password and regained access to the online banking account.


# Part III — Tokenization

Tokenization answers a deceptively important question:

> **What units should the NLP system operate on?**

For English text, tokens are often approximately words and punctuation marks.

But choices arise around:

- contractions,
- punctuation,
- numbers,
- currencies,
- emojis,
- hyphenated terms,
- URLs,
- usernames,
- domain-specific phrases.

Tokenization therefore affects every downstream step.

## Sentence Tokenization

Sentence tokenization splits a document into sentence units.

A naive strategy:

```python
text.split(".")
```

fails on abbreviations such as:

```text
Dr.
Mr.
Ltd.
p.m.
```

NLTK's Punkt tokenizer uses statistical punctuation patterns and heuristics.

In [4]:
text = "Dr. Tan works at UOB Ltd. He called Mr. Lee at 5 p.m. today."

print("Naive split:")
print([x.strip() for x in text.split(".") if x.strip()])

print("\nNLTK sentence tokenizer:")
print(sent_tokenize(text))

Naive split:
['Dr', 'Tan works at UOB Ltd', 'He called Mr', 'Lee at 5 p', 'm', 'today']

NLTK sentence tokenizer:
['Dr. Tan works at UOB Ltd.', 'He called Mr. Lee at 5 p.m. today.']


### Experiment — Sentence Boundary Edge Cases

In [5]:
examples = [
    "Dr. Tan arrived at 8 p.m. He called the customer.",
    "The payment failed. Please retry.",
    "Mr. Lee works at UOB Ltd. in Singapore.",
]

for text in examples:
    print("TEXT:", text)
    print("SENTENCES:", sent_tokenize(text))
    print("-" * 80)

TEXT: Dr. Tan arrived at 8 p.m. He called the customer.
SENTENCES: ['Dr. Tan arrived at 8 p.m.', 'He called the customer.']
--------------------------------------------------------------------------------
TEXT: The payment failed. Please retry.
SENTENCES: ['The payment failed.', 'Please retry.']
--------------------------------------------------------------------------------
TEXT: Mr. Lee works at UOB Ltd. in Singapore.
SENTENCES: ['Mr. Lee works at UOB Ltd. in Singapore.']
--------------------------------------------------------------------------------


### Computational intuition

For text length \(T\), sentence segmentation is approximately:

\[
\Theta(T)
\]

because the algorithm primarily scans the sequence.

The challenging issue is not computational complexity but **linguistic ambiguity**.

## Word Tokenization

Different tokenizers encode different assumptions.

We compare:

- `word_tokenize`
- `TreebankWordTokenizer`
- `RegexpTokenizer`
- `TweetTokenizer`

In [6]:
sample = "I can't approve SGD 120.50 via mobile-banking today!"

tokenizers = {
    "word_tokenize": lambda x: word_tokenize(x),
    "Treebank": TreebankWordTokenizer().tokenize,
    "Regexp": RegexpTokenizer(r"[A-Za-z]+|\d+(?:\.\d+)?").tokenize,
    "Tweet": TweetTokenizer().tokenize,
}

for name, tokenizer in tokenizers.items():
    print("\n", name)
    print(tokenizer(sample))


 word_tokenize
['I', 'ca', "n't", 'approve', 'SGD', '120.50', 'via', 'mobile-banking', 'today', '!']

 Treebank
['I', 'ca', "n't", 'approve', 'SGD', '120.50', 'via', 'mobile-banking', 'today', '!']

 Regexp
['I', 'can', 't', 'approve', 'SGD', '120.50', 'via', 'mobile', 'banking', 'today']

 Tweet
['I', "can't", 'approve', 'SGD', '120.50', 'via', 'mobile-banking', 'today', '!']


### Experiment — Tokenizer Choice vs Vocabulary Size

A tokenizer that creates more unique tokens may produce:

- larger feature spaces,
- more sparsity,
- more lexical detail,
- potentially more noise.

In [7]:
for name, tokenizer in tokenizers.items():
    toks = tokenizer(raw_text)

    print(
        f"{name:15s}",
        "tokens =", len(toks),
        "unique =", len(set(toks)),
    )

word_tokenize   tokens = 162 unique = 100
Treebank        tokens = 150 unique = 102
Regexp          tokens = 144 unique = 97
Tweet           tokens = 161 unique = 100


## Multi-Word Expression Tokenization

Sometimes known phrases should behave as one logical token:

```text
credit card → credit_card
bank account → bank_account
mobile payment → mobile_payment
```

NLTK provides `MWETokenizer`.

In [8]:
mwe = MWETokenizer(
    [
        ("credit", "card"),
        ("bank", "account"),
        ("mobile", "payment"),
    ],
    separator="_",
)

example = "someone used my credit card and bank account".split()

print(mwe.tokenize(example))

['someone', 'used', 'my', 'credit_card', 'and', 'bank_account']


# Part IV — Stopword Removal

Stopwords are frequently occurring function words such as:

```text
the
a
an
is
and
of
```

Removing them may improve some:

- frequency analyses,
- topic models,
- bag-of-words classifiers.

But stopwords are not universally meaningless.

In [9]:
stop_words = set(stopwords.words("english"))

text = "I did not approve this payment"

tokens0 = [
    token.lower()
    for token in word_tokenize(text)
    if token.isalpha()
]

filtered = [
    token
    for token in tokens0
    if token not in stop_words
]

print("Original:", tokens0)
print("Filtered:", filtered)

Original: ['i', 'did', 'not', 'approve', 'this', 'payment']
Filtered: ['approve', 'payment']


## Why Negation Matters

Notice that `"not"` can disappear.

That can transform:

```text
I did not approve payment
```

into something resembling:

```text
approve payment
```

which is semantically dangerous.

### Experiment — Custom Stopword Policy

In [10]:
custom_stopwords = stop_words - {
    "not",
    "no",
    "nor",
}

default_result = [
    t for t in tokens0
    if t not in stop_words
]

custom_result = [
    t for t in tokens0
    if t not in custom_stopwords
]

print("Default stopwords:", default_result)
print("Negation preserved:", custom_result)

Default stopwords: ['approve', 'payment']
Negation preserved: ['not', 'approve', 'payment']


### Tuning Strategy

Compare:

1. no stopword removal,
2. default NLTK stopwords,
3. domain-specific stopwords,
4. negation-preserving stopwords.

For sentiment and intent classification, option 1 or 4 is often worth testing first.

# Part V — Stemming

Stemming uses morphological heuristics to strip affixes.

Example:

```text
connecting
connected
connections

↓
connect
```

Stemming does not guarantee valid dictionary words.

Its main purpose is **feature normalization**.

In [11]:
words = [
    "connection",
    "connections",
    "connected",
    "connecting",
    "organization",
    "organizational",
    "studies",
    "studying",
]

stemmers = {
    "Porter": PorterStemmer(),
    "Lancaster": LancasterStemmer(),
    "Snowball": SnowballStemmer("english"),
}

for word in words:
    print(
        f"{word:15s}",
        {
            name: stemmer.stem(word)
            for name, stemmer in stemmers.items()
        }
    )

connection      {'Porter': 'connect', 'Lancaster': 'connect', 'Snowball': 'connect'}
connections     {'Porter': 'connect', 'Lancaster': 'connect', 'Snowball': 'connect'}
connected       {'Porter': 'connect', 'Lancaster': 'connect', 'Snowball': 'connect'}
connecting      {'Porter': 'connect', 'Lancaster': 'connect', 'Snowball': 'connect'}
organization    {'Porter': 'organ', 'Lancaster': 'org', 'Snowball': 'organ'}
organizational  {'Porter': 'organiz', 'Lancaster': 'org', 'Snowball': 'organiz'}
studies         {'Porter': 'studi', 'Lancaster': 'study', 'Snowball': 'studi'}
studying        {'Porter': 'studi', 'Lancaster': 'study', 'Snowball': 'studi'}


## Experiment — Aggressive vs Conservative Stemming

Lancaster is generally more aggressive than Porter.

Aggressive stemming may:

- reduce vocabulary,
- improve recall,
- but collapse unrelated terms.

In [12]:
for word in [
    "maximum",
    "multiply",
    "university",
    "universe",
    "policy",
    "police",
]:
    print(
        f"{word:12s}",
        "Porter =", PorterStemmer().stem(word),
        "Lancaster =", LancasterStemmer().stem(word),
    )

maximum      Porter = maximum Lancaster = maxim
multiply     Porter = multipli Lancaster = multiply
university   Porter = univers Lancaster = univers
universe     Porter = univers Lancaster = univers
policy       Porter = polici Lancaster = policy
police       Porter = polic Lancaster = pol


# Part VI — Lemmatization

Lemmatization attempts to map words to dictionary forms.

Examples:

```text
cars → car
running → run
better → good
```

Unlike stemming, lemmatization can use grammatical information.

In [13]:
lemmatizer = WordNetLemmatizer()

examples = [
    ("cars", "n"),
    ("running", "v"),
    ("better", "a"),
    ("went", "v"),
]

for word, pos in examples:
    print(
        f"{word:10s}",
        "default =", lemmatizer.lemmatize(word),
        "POS-aware =", lemmatizer.lemmatize(word, pos=pos),
    )

cars       default = car POS-aware = car
running    default = running POS-aware = run
better     default = better POS-aware = good
went       default = went POS-aware = go


## POS-Aware Lemmatization

NLTK's POS tagger emits Penn Treebank tags such as:

```text
NN
VB
JJ
RB
```

WordNet expects:

```text
n
v
a
r
```

So we create a mapping.

In [53]:
def penn_to_wordnet(tag):
    if tag.startswith("J"):
        return wn.ADJ

    if tag.startswith("V"):
        return wn.VERB

    if tag.startswith("N"):
        return wn.NOUN

    if tag.startswith("R"):
        return wn.ADV

    return wn.NOUN

In [54]:
sentence = "Customers were running better fraud detection systems"

tagged = pos_tag(word_tokenize(sentence))

lemmatized = [
    (
        word,
        tag,
        lemmatizer.lemmatize(
            word.lower(),
            penn_to_wordnet(tag),
        ),
    )
    for word, tag in tagged
    if word.isalpha()
]

pprint(lemmatized)

[('Customers', 'NNS', 'customer'),
 ('were', 'VBD', 'be'),
 ('running', 'VBG', 'run'),
 ('better', 'RB', 'well'),
 ('fraud', 'NN', 'fraud'),
 ('detection', 'NN', 'detection'),
 ('systems', 'NNS', 'system')]


# Part VII — Part-of-Speech Tagging

POS tagging assigns grammatical categories to tokens.

Examples:

```text
transaction → NN
suspicious → JJ
reported → VBD
quickly → RB
```

POS tagging helps:

- lemmatization,
- chunking,
- parsing,
- information extraction,
- rule-based NLP.

In [55]:
sentence = "The suspicious card transaction appeared on my monthly statement."

pprint(
    pos_tag(
        word_tokenize(sentence)
    )
)

[('The', 'DT'),
 ('suspicious', 'JJ'),
 ('card', 'JJ'),
 ('transaction', 'NN'),
 ('appeared', 'VBD'),
 ('on', 'IN'),
 ('my', 'PRP$'),
 ('monthly', 'JJ'),
 ('statement', 'NN'),
 ('.', '.')]


## Experiment — Same Word, Different POS

Words can change grammatical role depending on context.

In [56]:
examples = [
    "The customer checked the record.",
    "Please record the transaction.",
    "The bank will transfer funds.",
    "The transfer was successful.",
]

for sentence in examples:
    print(sentence)
    pprint(pos_tag(word_tokenize(sentence)))
    print()

The customer checked the record.
[('The', 'DT'),
 ('customer', 'NN'),
 ('checked', 'VBD'),
 ('the', 'DT'),
 ('record', 'NN'),
 ('.', '.')]

Please record the transaction.
[('Please', 'NN'),
 ('record', 'NN'),
 ('the', 'DT'),
 ('transaction', 'NN'),
 ('.', '.')]

The bank will transfer funds.
[('The', 'DT'),
 ('bank', 'NN'),
 ('will', 'MD'),
 ('transfer', 'VB'),
 ('funds', 'NNS'),
 ('.', '.')]

The transfer was successful.
[('The', 'DT'),
 ('transfer', 'NN'),
 ('was', 'VBD'),
 ('successful', 'JJ'),
 ('.', '.')]



## Experiment — Domain Shift

Taggers trained on general-domain English may behave differently on technical terminology.

In [18]:
domain_sentences = [
    "KYC remediation failed overnight.",
    "AML alerts surged after batch reconciliation.",
    "UAT deployment broke downstream ETL jobs.",
]

for sentence in domain_sentences:
    print(sentence)
    print(pos_tag(word_tokenize(sentence)))
    print()

KYC remediation failed overnight.
[('KYC', 'NNP'), ('remediation', 'NN'), ('failed', 'VBD'), ('overnight', 'RB'), ('.', '.')]

AML alerts surged after batch reconciliation.
[('AML', 'NNP'), ('alerts', 'NNS'), ('surged', 'VBD'), ('after', 'IN'), ('batch', 'NN'), ('reconciliation', 'NN'), ('.', '.')]

UAT deployment broke downstream ETL jobs.
[('UAT', 'NNP'), ('deployment', 'NN'), ('broke', 'VBD'), ('downstream', 'JJ'), ('ETL', 'NNP'), ('jobs', 'NNS'), ('.', '.')]



# Part VIII — Chunking / Shallow Parsing

Chunking groups POS-tagged tokens into phrase-level units.

It is called **shallow parsing** because it does not necessarily recover the complete syntactic tree.

A common target is the noun phrase:

```text
the suspicious card transaction
```

In [19]:
sentence = "The suspicious card transaction appeared on my monthly statement."

tagged = pos_tag(
    word_tokenize(sentence)
)

print(tagged)

[('The', 'DT'), ('suspicious', 'JJ'), ('card', 'JJ'), ('transaction', 'NN'), ('appeared', 'VBD'), ('on', 'IN'), ('my', 'PRP$'), ('monthly', 'JJ'), ('statement', 'NN'), ('.', '.')]


## Experiment 1 — Gradually Broaden the Chunk Grammar

### Grammar A
Only nouns.

### Grammar B
Adjectives followed by nouns.

### Grammar C
Optional determiner + adjectives + one or more nouns.

In [57]:
grammars = {
    "A_noun_only":
        "NP: {<NN.*>+}",

    "B_adj_noun":
        "NP: {<JJ.*>*<NN.*>+}",

    "C_full_np":
        "NP: {<DT|PRP\\$>?<JJ.*>*<NN.*>+}",
}

for name, grammar in grammars.items():

    parser = RegexpParser(grammar)

    print("\n", name)
    print(parser.parse(tagged))


 A_noun_only
(S
  (NP Customers/NNS)
  were/VBD
  running/VBG
  better/RB
  (NP fraud/NN detection/NN systems/NNS))

 B_adj_noun
(S
  (NP Customers/NNS)
  were/VBD
  running/VBG
  better/RB
  (NP fraud/NN detection/NN systems/NNS))

 C_full_np
(S
  (NP Customers/NNS)
  were/VBD
  running/VBG
  better/RB
  (NP fraud/NN detection/NN systems/NNS))


### Interpretation

As the grammar becomes more permissive:

- phrase recall increases,
- phrase length increases,
- false merges may increase.

This demonstrates the same **precision–recall trade-off** seen in statistical models.

## Experiment 2 — Extract Noun Phrases Programmatically

In [58]:
np_parser = RegexpParser(
    "NP: {<DT|PRP\\$>?<JJ.*>*<NN.*>+}"
)

def extract_noun_phrases(sentence):

    tagged = pos_tag(
        word_tokenize(sentence)
    )

    tree = np_parser.parse(tagged)

    return [
        " ".join(
            word
            for word, tag
            in subtree.leaves()
        )
        for subtree
        in tree.subtrees(
            lambda t: t.label() == "NP"
        )
    ]

for doc in documents[:8]:

    print(doc["text"])
    print(
        "NPs:",
        extract_noun_phrases(doc["text"])
    )
    print()

Arjun paid SGD 120 at Marina Bay yesterday, but the card payment was declined twice.
NPs: ['Arjun', 'SGD', 'Marina Bay yesterday', 'the card payment']

Priya cannot access her savings account after resetting the password.
NPs: ['Priya', 'access', 'her savings', 'the password']

Daniel noticed an unauthorized transaction from London and wants the card blocked immediately.
NPs: ['Daniel', 'an unauthorized transaction', 'London', 'the card']

The mobile payment worked after I restarted the app. Excellent support!
NPs: ['The mobile payment', 'the app', 'Excellent support']

My bank account is locked and login verification keeps failing.
NPs: ['My bank account', 'login verification']

Someone used my credit card online. I did not make this purchase.
NPs: ['Someone', 'my credit card online', 'this purchase']

The transfer to Singapore was completed successfully, but the merchant has not received it.
NPs: ['The transfer', 'Singapore', 'the merchant']

Password reset was quick and the account 

## Experiment 3 — Add Verb Phrases

Chunk grammars can contain multiple phrase types.

In [59]:
grammar = (
    "NP: {<DT|PRP\\$>?<JJ.*>*<NN.*>+}\n"
    "VP: {<VB.*><RB.*>*<VB.*>*}"
)

parser = RegexpParser(grammar)

sentence = (
    "The suspicious transaction "
    "was immediately blocked."
)

print(
    parser.parse(
        pos_tag(
            word_tokenize(sentence)
        )
    )
)

(S
  (NP The/DT suspicious/JJ transaction/NN)
  (VP was/VBD immediately/RB blocked/VBN)
  ./.)


## How to Tune a Rule-Based Chunker

A practical workflow:

1. label a small validation dataset,
2. start with strict patterns,
3. inspect false negatives,
4. broaden the grammar,
5. inspect false positives,
6. add exclusions,
7. repeat.

This is effectively **manual symbolic model tuning**.

# Part IX — Named Entity Recognition

NER identifies spans representing entities such as:

- PERSON,
- ORGANIZATION,
- GPE,
- LOCATION.

NLTK includes a classical statistical named-entity chunker.

In [60]:
sentences = [
    "Daniel travelled from Singapore to London.",
    "Arjun visited Marina Bay.",
    "Microsoft opened an office in Singapore.",
]

for sentence in sentences:

    tagged = pos_tag(
        word_tokenize(sentence)
    )

    print("\n", sentence)
    print(ne_chunk(tagged))


 Daniel travelled from Singapore to London.
(S
  (PERSON Daniel/NNP)
  travelled/VBD
  from/IN
  (GPE Singapore/NNP)
  to/TO
  (GPE London/NNP)
  ./.)

 Arjun visited Marina Bay.
(S (PERSON Arjun/NNP) visited/VBD (PERSON Marina/NNP Bay/NNP) ./.)

 Microsoft opened an office in Singapore.
(S
  (PERSON Microsoft/NNP)
  opened/VBD
  an/DT
  office/NN
  in/IN
  (GPE Singapore/NNP)
  ./.)


## Experiment — Binary vs Multiclass NER

`binary=True` collapses all named entity types into a single generic `NE`.

In [61]:
sentence = (
    "Daniel travelled from Singapore "
    "to London and contacted Microsoft."
)

tagged = pos_tag(
    word_tokenize(sentence)
)

print("Multiclass")
print(ne_chunk(tagged, binary=False))

print("\nBinary")
print(ne_chunk(tagged, binary=True))

Multiclass
(S
  (PERSON Daniel/NNP)
  travelled/VBD
  from/IN
  (GPE Singapore/NNP)
  to/TO
  (GPE London/NNP)
  and/CC
  contacted/VBN
  (PERSON Microsoft/NNP)
  ./.)

Binary
(S
  (NE Daniel/NNP)
  travelled/VBD
  from/IN
  (NE Singapore/NNP)
  to/TO
  (NE London/NNP)
  and/CC
  contacted/VBN
  (NE Microsoft/NNP)
  ./.)


## Convert NER Tree into Structured Records

In [62]:
def extract_entities(sentence):

    tree = ne_chunk(
        pos_tag(
            word_tokenize(sentence)
        )
    )

    entities = []

    for node in tree:

        if hasattr(node, "label"):

            entities.append(
                {
                    "text": " ".join(
                        token
                        for token, tag
                        in node.leaves()
                    ),
                    "label": node.label(),
                }
            )

    return entities


for doc in documents:

    entities = extract_entities(
        doc["text"]
    )

    if entities:
        print(
            doc["id"],
            entities
        )

1 [{'text': 'Arjun', 'label': 'PERSON'}, {'text': 'SGD', 'label': 'ORGANIZATION'}, {'text': 'Marina Bay', 'label': 'FACILITY'}]
3 [{'text': 'Daniel', 'label': 'PERSON'}, {'text': 'London', 'label': 'GPE'}]
6 [{'text': 'Someone', 'label': 'PERSON'}]
7 [{'text': 'Singapore', 'label': 'GPE'}]
8 [{'text': 'Password', 'label': 'GPE'}]
10 [{'text': 'New York', 'label': 'GPE'}]


## Failure Experiment — Domain-Specific Entities

General NER systems may struggle with:

- proprietary product names,
- unusual capitalization,
- abbreviations,
- domain-specific organizations.

In [63]:
examples = [
    "AMLShield raised an alert for PayNowX.",
    "FinGuard detected suspicious activity in ACMEPay.",
    "uob mobile banking failed after login.",
]

for sentence in examples:

    print(sentence)
    print(
        extract_entities(sentence)
    )
    print()

AMLShield raised an alert for PayNowX.
[{'text': 'AMLShield', 'label': 'ORGANIZATION'}, {'text': 'PayNowX', 'label': 'ORGANIZATION'}]

FinGuard detected suspicious activity in ACMEPay.
[{'text': 'FinGuard', 'label': 'ORGANIZATION'}, {'text': 'ACMEPay', 'label': 'GPE'}]

uob mobile banking failed after login.
[]



### Production Perspective

NLTK NER is excellent for learning classical NER mechanics.

For modern production NER, consider:

- spaCy,
- Hugging Face transformers,
- Flair,
- domain-fine-tuned transformer models.

# Part X — Context-Free Grammars and Constituency Parsing

A Context-Free Grammar defines syntactic production rules.

For example:

```text
S → NP VP
NP → Det N
VP → V NP
```

A parser tries to determine whether a sequence can be generated by the grammar and constructs the corresponding tree.

In [64]:
grammar = CFG.fromstring(
    "S -> NP VP\n"
    "NP -> Det N | Det Adj N | ProperNoun\n"
    "VP -> V NP\n"
    "Det -> 'the' | 'a'\n"
    "Adj -> 'suspicious'\n"
    "N -> 'customer' | 'transaction' | 'account'\n"
    "ProperNoun -> 'Daniel'\n"
    "V -> 'reported'"
)

parser = ChartParser(grammar)

sentence = (
    "Daniel reported a suspicious transaction"
    .split()
)

for tree in parser.parse(sentence):
    print(tree)

(S
  (NP (ProperNoun Daniel))
  (VP (V reported) (NP (Det a) (Adj suspicious) (N transaction))))


## Experiment — Grammar Coverage

Symbolic grammars cannot automatically generalize to missing rules or vocabulary.

In [65]:
sentences = [
    "Daniel reported a suspicious transaction".split(),
    "Daniel reported the account".split(),
]

for sentence in sentences:

    print("\nSentence:", sentence)

    try:
        trees = list(
            parser.parse(sentence)
        )

        print(
            "Number of parses:",
            len(trees)
        )

        for tree in trees:
            print(tree)

    except ValueError as exc:
        print(
            "Parser error:",
            exc
        )


Sentence: ['Daniel', 'reported', 'a', 'suspicious', 'transaction']
Number of parses: 1
(S
  (NP (ProperNoun Daniel))
  (VP (V reported) (NP (Det a) (Adj suspicious) (N transaction))))

Sentence: ['Daniel', 'reported', 'the', 'account']
Number of parses: 1
(S
  (NP (ProperNoun Daniel))
  (VP (V reported) (NP (Det the) (N account))))


## Experiment — Expand the Grammar

We explicitly add vocabulary and syntactic alternatives.

In [66]:
grammar_v2 = CFG.fromstring(
    "S -> NP VP\n"
    "NP -> Det N | Det Adj N | ProperNoun\n"
    "VP -> V NP\n"
    "Det -> 'the' | 'a'\n"
    "Adj -> 'suspicious' | 'locked'\n"
    "N -> 'customer' | 'transaction' | 'account' | 'password'\n"
    "ProperNoun -> 'Daniel'\n"
    "V -> 'reported' | 'reset'"
)

parser_v2 = ChartParser(
    grammar_v2
)

for sentence in [
    "Daniel reported the account".split(),
    "Daniel reset the password".split(),
]:

    print("\n", sentence)

    for tree in parser_v2.parse(
        sentence
    ):
        print(tree)


 ['Daniel', 'reported', 'the', 'account']
(S
  (NP (ProperNoun Daniel))
  (VP (V reported) (NP (Det the) (N account))))

 ['Daniel', 'reset', 'the', 'password']
(S
  (NP (ProperNoun Daniel))
  (VP (V reset) (NP (Det the) (N password))))


## Constituency Tree Inspection

In [67]:
tree = next(
    parser_v2.parse(
        "Daniel reset the password".split()
    )
)

print(tree)

print(
    "Tree height:",
    tree.height()
)

print(
    "Leaves:",
    tree.leaves()
)

print("\nNoun phrases:")

for subtree in tree.subtrees(
    lambda t: t.label() == "NP"
):
    print(subtree)

(S
  (NP (ProperNoun Daniel))
  (VP (V reset) (NP (Det the) (N password))))
Tree height: 5
Leaves: ['Daniel', 'reset', 'the', 'password']

Noun phrases:
(NP (ProperNoun Daniel))
(NP (Det the) (N password))


# Part XI — Probabilistic Context-Free Grammars

A PCFG augments grammar rules with probabilities.

Example:

```text
NP → Det N [0.6]
NP → ProperNoun [0.4]
```

The probability of a parse is derived from the probabilities of the grammar rules used.

Probabilities rank alternative parses.

In [68]:
pcfg = PCFG.fromstring(
    "S -> NP VP [1.0]\n"
    "NP -> ProperNoun [0.30]\n"
    "NP -> Det N [0.40]\n"
    "NP -> Det Adj N [0.30]\n"
    "VP -> V NP [1.0]\n"
    "Det -> 'the' [0.6]\n"
    "Det -> 'a' [0.4]\n"
    "Adj -> 'suspicious' [0.7]\n"
    "Adj -> 'locked' [0.3]\n"
    "N -> 'transaction' [0.4]\n"
    "N -> 'account' [0.3]\n"
    "N -> 'password' [0.3]\n"
    "ProperNoun -> 'Daniel' [1.0]\n"
    "V -> 'reported' [0.6]\n"
    "V -> 'reset' [0.4]"
)

viterbi = ViterbiParser(
    pcfg
)

for tree in viterbi.parse(
    "Daniel reported a suspicious transaction".split()
):

    print(tree)

    print(
        "Probability:",
        tree.prob()
    )

(S
  (NP (ProperNoun Daniel))
  (VP
    (V reported)
    (NP (Det a) (Adj suspicious) (N transaction)))) (p=0.006048)
Probability: 0.006047999999999999


## Tuning Insight

Changing PCFG probabilities:

- does **not** make illegal parses legal,
- changes the ranking among legal parses.

This is the bridge from symbolic grammar toward statistical parsing.

# Part XII — Dependency Structures

Constituency parsing groups words into phrases.

Dependency parsing focuses on **head-dependent relations**.

Example:

```text
reported
├── nsubj → Daniel
└── obj   → transaction
```

NLTK can represent dependency graphs and work with dependency annotations.

In [69]:
dependency_data = (
    "Daniel NNP 2 nsubj\n"
    "reported VBD 0 root\n"
    "transaction NN 2 obj\n"
)

graph = DependencyGraph(
    dependency_data
)

for address, node in graph.nodes.items():

    if address == 0:
        continue

    print(
        {
            "word": node["word"],
            "head": node["head"],
            "relation": node["rel"],
            "dependents": node["deps"],
        }
    )

{'word': 'Daniel', 'head': 2, 'relation': 'nsubj', 'dependents': defaultdict(<class 'list'>, {})}
{'word': 'reported', 'head': 0, 'relation': 'root', 'dependents': defaultdict(<class 'list'>, {'nsubj': [1], 'obj': [3]})}
{'word': 'transaction', 'head': 2, 'relation': 'obj', 'dependents': defaultdict(<class 'list'>, {})}


### Important Limitation

NLTK provides dependency data structures and interfaces, but for high-quality modern dependency parsing use:

- spaCy,
- Stanza,
- transformer-based parsers.

# Part XIII — WordNet and Lexical Semantics

WordNet is one of NLTK's most important lexical resources.

It organizes concepts into **synsets**.

A word can have several meanings.

Example:

```text
bank
├── financial institution
├── river edge
├── slope
└── reserve/storage sense
```

In [33]:
for synset in wn.synsets("bank")[:8]:

    print(
        synset.name(),
        "=>",
        synset.definition()
    )

bank.n.01 => sloping land (especially the slope beside a body of water)
depository_financial_institution.n.01 => a financial institution that accepts deposits and channels the money into lending activities
bank.n.03 => a long ridge or pile
bank.n.04 => an arrangement of similar objects in a row or in tiers
bank.n.05 => a supply or stock held in reserve for future use (especially in emergencies)
bank.n.06 => the funds held by a gambling house or the dealer in some gambling games
bank.n.07 => a slope in the turn of a road or track; the outside is higher than the inside in order to reduce the effects of centrifugal force
savings_bank.n.02 => a container (usually with a slot in the top) for keeping money at home


## Synonyms and Antonyms

In [70]:
synonyms = set()
antonyms = set()

for synset in wn.synsets("good"):

    for lemma in synset.lemmas():

        synonyms.add(
            lemma.name()
        )

        antonyms.update(
            x.name()
            for x in lemma.antonyms()
        )

print(
    "Some synonyms:",
    sorted(synonyms)[:20]
)

print(
    "Antonyms:",
    sorted(antonyms)
)

Some synonyms: ['adept', 'beneficial', 'commodity', 'dear', 'dependable', 'effective', 'estimable', 'expert', 'full', 'good', 'goodness', 'honest', 'honorable', 'in_effect', 'in_force', 'just', 'near', 'practiced', 'proficient', 'respectable']
Antonyms: ['bad', 'badness', 'evil', 'evilness', 'ill']


## Hypernyms and Hyponyms

A **hypernym** is a broader concept.

```text
dog → canine → carnivore → mammal → animal
```

A **hyponym** is a more specific concept.

In [71]:
dog = wn.synset(
    "dog.n.01"
)

print(
    "Hypernyms:",
    dog.hypernyms()
)

print(
    "Some hyponyms:",
    dog.hyponyms()[:10]
)

print(
    "Part meronyms:",
    dog.part_meronyms()
)

Hypernyms: [Synset('domestic_animal.n.01'), Synset('canine.n.02')]
Some hyponyms: [Synset('hunting_dog.n.01'), Synset('newfoundland.n.01'), Synset('corgi.n.01'), Synset('leonberg.n.01'), Synset('cur.n.01'), Synset('pug.n.01'), Synset('dalmatian.n.02'), Synset('pooch.n.01'), Synset('lapdog.n.01'), Synset('spitz.n.01')]
Part meronyms: [Synset('flag.n.07')]


# Part XIV — WordNet Semantic Similarity

WordNet supports graph-based semantic similarity.

Common metrics include:

- path similarity,
- Wu-Palmer similarity,
- Leacock-Chodorow similarity,
- Resnik similarity,
- Lin similarity,
- Jiang-Conrath similarity.

Different metrics encode different assumptions.

In [36]:
pairs = [
    (
        "dog.n.01",
        "cat.n.01",
    ),
    (
        "dog.n.01",
        "car.n.01",
    ),
    (
        "bank.n.01",
        "depository_financial_institution.n.01",
    ),
]

for a, b in pairs:

    s1 = wn.synset(a)
    s2 = wn.synset(b)

    print("\n", a, "vs", b)

    print(
        "path:",
        s1.path_similarity(s2)
    )

    print(
        "Wu-Palmer:",
        s1.wup_similarity(s2)
    )

    try:
        print(
            "LCH:",
            s1.lch_similarity(s2)
        )

    except Exception as exc:
        print(
            "LCH unavailable:",
            exc
        )


 dog.n.01 vs cat.n.01
path: 0.2
Wu-Palmer: 0.8571428571428571
LCH: 2.0281482472922856

 dog.n.01 vs car.n.01
path: 0.07692307692307693
Wu-Palmer: 0.4
LCH: 1.072636802264849

 bank.n.01 vs depository_financial_institution.n.01
path: 0.07692307692307693
Wu-Palmer: 0.14285714285714285
LCH: 1.072636802264849


## Experiment — Sense Choice Changes Similarity

The word `"bank"` has many synsets.

Similarity should therefore compare **meanings**, not raw strings.

In [37]:
bank_senses = wn.synsets(
    "bank",
    pos=wn.NOUN
)

money = wn.synset(
    "money.n.01"
)

river = wn.synset(
    "river.n.01"
)

for sense in bank_senses[:6]:

    print(
        "\n",
        sense.name(),
        "=>",
        sense.definition()
    )

    print(
        "similarity to money:",
        sense.wup_similarity(money)
    )

    print(
        "similarity to river:",
        sense.wup_similarity(river)
    )


 bank.n.01 => sloping land (especially the slope beside a body of water)
similarity to money: 0.15384615384615385
similarity to river: 0.3333333333333333

 depository_financial_institution.n.01 => a financial institution that accepts deposits and channels the money into lending activities
similarity to money: 0.26666666666666666
similarity to river: 0.14285714285714285

 bank.n.03 => a long ridge or pile
similarity to money: 0.14285714285714285
similarity to river: 0.3076923076923077

 bank.n.04 => an arrangement of similar objects in a row or in tiers
similarity to money: 0.3076923076923077
similarity to river: 0.16666666666666666

 bank.n.05 => a supply or stock held in reserve for future use (especially in emergencies)
similarity to money: 0.2
similarity to river: 0.10526315789473684

 bank.n.06 => the funds held by a gambling house or the dealer in some gambling games
similarity to money: 0.2857142857142857
similarity to river: 0.15384615384615385


# Part XV — Word Sense Disambiguation

WSD determines which meaning of an ambiguous word is active in a sentence.

Example:

```text
I deposited money in the bank.
```

versus:

```text
I sat on the river bank.
```

NLTK provides the classical **Lesk algorithm**.

In [38]:
examples = [
    "I deposited money into the bank",
    "We sat on the bank of the river",
    "The bank approved the loan",
    "The fisherman stood on the bank",
]

for sentence in examples:

    sense = lesk(
        word_tokenize(
            sentence.lower()
        ),
        "bank",
    )

    print(
        "\nSentence:",
        sentence
    )

    print(
        "Sense:",
        sense
    )

    if sense:
        print(
            "Definition:",
            sense.definition()
        )


Sentence: I deposited money into the bank
Sense: Synset('depository_financial_institution.n.01')
Definition: a financial institution that accepts deposits and channels the money into lending activities

Sentence: We sat on the bank of the river
Sense: Synset('bank.n.01')
Definition: sloping land (especially the slope beside a body of water)

Sentence: The bank approved the loan
Sense: Synset('bank.n.01')
Definition: sloping land (especially the slope beside a body of water)

Sentence: The fisherman stood on the bank
Sense: Synset('bank.n.01')
Definition: sloping land (especially the slope beside a body of water)


## Experiment — Strengthen Context

Lesk relies heavily on overlap between context words and dictionary definitions.

Adding more context may change the predicted sense.

In [39]:
contexts = [
    "bank",
    "money bank",
    "money loan deposit bank",
    "river water shore bank",
    "fishing river water bank",
]

for context in contexts:

    sense = lesk(
        word_tokenize(context),
        "bank",
    )

    print(
        f"{context:30s}",
        "=>",
        sense.name()
        if sense else None
    )

bank                           => bank.v.02
money bank                     => depository_financial_institution.n.01
money loan deposit bank        => depository_financial_institution.n.01
river water shore bank         => bank.v.02
fishing river water bank       => bank.v.02


# Part XVI — N-Grams

An N-gram is a contiguous sequence of \(N\) tokens.

Examples:

```text
unigram:
card

bigram:
credit card

trigram:
credit card payment
```

N-grams are not one algorithm.

They are a reusable representation used for:

- feature extraction,
- phrase detection,
- collocations,
- language modelling,
- statistical NLP.

In [40]:
tokens = [
    token.lower()
    for token in word_tokenize(raw_text)
    if token.isalpha()
]

print(
    "Unigrams:",
    tokens[:12]
)

print(
    "Bigrams:",
    list(bigrams(tokens))[:10]
)

print(
    "Trigrams:",
    list(trigrams(tokens))[:10]
)

print(
    "4-grams:",
    list(ngrams(tokens, 4))[:5]
)

Unigrams: ['arjun', 'paid', 'sgd', 'at', 'marina', 'bay', 'yesterday', 'but', 'the', 'card', 'payment', 'was']
Bigrams: [('arjun', 'paid'), ('paid', 'sgd'), ('sgd', 'at'), ('at', 'marina'), ('marina', 'bay'), ('bay', 'yesterday'), ('yesterday', 'but'), ('but', 'the'), ('the', 'card'), ('card', 'payment')]
Trigrams: [('arjun', 'paid', 'sgd'), ('paid', 'sgd', 'at'), ('sgd', 'at', 'marina'), ('at', 'marina', 'bay'), ('marina', 'bay', 'yesterday'), ('bay', 'yesterday', 'but'), ('yesterday', 'but', 'the'), ('but', 'the', 'card'), ('the', 'card', 'payment'), ('card', 'payment', 'was')]
4-grams: [('arjun', 'paid', 'sgd', 'at'), ('paid', 'sgd', 'at', 'marina'), ('sgd', 'at', 'marina', 'bay'), ('at', 'marina', 'bay', 'yesterday'), ('marina', 'bay', 'yesterday', 'but')]


## Complexity

For fixed \(N\), generating N-grams over \(T\) tokens takes approximately:

\[
\Theta(T)
\]

The real challenge is the number of **unique N-grams**.

If vocabulary size is \(V\), possible combinations grow rapidly:

\[
V^2,\quad V^3,\quad \ldots
\]

This creates sparsity.

# Part XVII — Frequency Analysis

`FreqDist` behaves like a frequency-aware dictionary.

Frequency asks:

> How often did this token occur?

It does not automatically tell us whether the token is informative.

In [41]:
fdist = FreqDist(tokens)

print(
    fdist.most_common(20)
)

print(
    "card frequency:",
    fdist["card"]
)

[('the', 16), ('card', 6), ('was', 4), ('not', 4), ('account', 4), ('and', 4), ('but', 3), ('payment', 3), ('after', 3), ('password', 3), ('my', 3), ('at', 2), ('access', 2), ('transaction', 2), ('i', 2), ('is', 2), ('online', 2), ('this', 2), ('to', 2), ('customer', 2)]
card frequency: 6


## Experiment — Raw vs Content-Word Frequency

Stopwords often dominate raw token counts.

In [42]:
sw = set(
    stopwords.words("english")
)

content_tokens = [
    token
    for token in tokens
    if token not in sw
]

print("RAW")
pprint(
    FreqDist(tokens).most_common(15)
)

print("\nCONTENT WORDS")
pprint(
    FreqDist(content_tokens).most_common(15)
)

RAW
[('the', 16),
 ('card', 6),
 ('was', 4),
 ('not', 4),
 ('account', 4),
 ('and', 4),
 ('but', 3),
 ('payment', 3),
 ('after', 3),
 ('password', 3),
 ('my', 3),
 ('at', 2),
 ('access', 2),
 ('transaction', 2),
 ('i', 2)]

CONTENT WORDS
[('card', 6),
 ('account', 4),
 ('payment', 3),
 ('password', 3),
 ('access', 2),
 ('transaction', 2),
 ('online', 2),
 ('customer', 2),
 ('arjun', 1),
 ('paid', 1),
 ('sgd', 1),
 ('marina', 1),
 ('bay', 1),
 ('yesterday', 1),
 ('declined', 1)]


# Part XVIII — Conditional Frequency Distributions

A Conditional Frequency Distribution stores:

```text
condition → frequency distribution
```

For this case study:

```text
support category → token frequencies
```

This can help identify vocabulary associated with different classes.

In [43]:
cfd = ConditionalFreqDist(
    (
        doc["label"],
        token.lower(),
    )
    for doc in documents
    for token in word_tokenize(
        doc["text"]
    )
    if token.isalpha()
)

for label in [
    "payment",
    "account",
    "fraud",
]:

    print(
        "\n",
        label
    )

    pprint(
        cfd[label].most_common(10)
    )


 payment
[('the', 8),
 ('but', 3),
 ('payment', 3),
 ('was', 3),
 ('at', 2),
 ('card', 2),
 ('not', 2),
 ('arjun', 1),
 ('paid', 1),
 ('sgd', 1)]

 account
[('the', 5),
 ('account', 4),
 ('password', 3),
 ('and', 3),
 ('access', 2),
 ('is', 2),
 ('priya', 1),
 ('can', 1),
 ('not', 1),
 ('her', 1)]

 fraud
[('card', 4),
 ('the', 3),
 ('transaction', 2),
 ('my', 2),
 ('this', 2),
 ('daniel', 1),
 ('noticed', 1),
 ('an', 1),
 ('unauthorized', 1),
 ('from', 1)]


## Experiment — Candidate Discriminative Terms

In [44]:
labels = [
    "payment",
    "account",
    "fraud",
]

candidate_words = set()

for label in labels:

    candidate_words.update(
        word
        for word, count
        in cfd[label].most_common(15)
    )

for word in sorted(
    candidate_words
):

    counts = {
        label: cfd[label][word]
        for label in labels
    }

    if max(
        counts.values()
    ) >= 2:
        print(
            word,
            counts
        )

access {'payment': 0, 'account': 2, 'fraud': 0}
account {'payment': 0, 'account': 4, 'fraud': 0}
and {'payment': 0, 'account': 3, 'fraud': 1}
at {'payment': 2, 'account': 0, 'fraud': 0}
but {'payment': 3, 'account': 0, 'fraud': 0}
card {'payment': 2, 'account': 0, 'fraud': 4}
is {'payment': 0, 'account': 2, 'fraud': 0}
my {'payment': 0, 'account': 1, 'fraud': 2}
not {'payment': 2, 'account': 1, 'fraud': 1}
password {'payment': 0, 'account': 3, 'fraud': 0}
payment {'payment': 3, 'account': 0, 'fraud': 0}
the {'payment': 8, 'account': 5, 'fraud': 3}
this {'payment': 0, 'account': 0, 'fraud': 2}
transaction {'payment': 0, 'account': 0, 'fraud': 2}
was {'payment': 3, 'account': 1, 'fraud': 0}


# Part XIX — Collocation Detection

A collocation is a sequence of words that occurs together more strongly than expected by chance.

Examples:

```text
credit card
mobile payment
machine learning
heavy rain
```

Raw frequency alone can be misleading.

One important association measure is **Pointwise Mutual Information**:

\[
PMI(x,y)
=
\log
\frac{P(x,y)}
{P(x)P(y)}
\]

High PMI means the pair occurs together more frequently than expected if the words were independent.

In [45]:
finder = BigramCollocationFinder.from_words(
    tokens
)

finder.apply_freq_filter(2)

pmi_scores = finder.score_ngrams(
    BigramAssocMeasures.pmi
)

for pair, score in pmi_scores[:15]:

    print(
        pair,
        round(score, 3)
    )

('account', 'is') 5.17
('card', 'payment') 4.0
('but', 'the') 3.17
('the', 'customer') 3.17
('the', 'card') 2.585
('the', 'password') 2.585


## Experiment 1 — Frequency Threshold

PMI tends to reward rare word pairs.

A minimum-frequency threshold can stabilize results.

In [46]:
for min_freq in [
    1,
    2,
    3,
]:

    finder = BigramCollocationFinder.from_words(
        tokens
    )

    finder.apply_freq_filter(
        min_freq
    )

    scores = finder.score_ngrams(
        BigramAssocMeasures.pmi
    )

    print(
        "\nmin_freq =",
        min_freq
    )

    for pair, score in scores[:8]:

        print(
            pair,
            round(score, 3)
        )


min_freq = 1
('a', 'suspicious') 7.17
('accessible', 'again') 7.17
('again', 'a') 7.17
('an', 'unauthorized') 7.17
('app', 'excellent') 7.17
('appeared', 'on') 7.17
('arjun', 'paid') 7.17
('bay', 'yesterday') 7.17

min_freq = 2
('account', 'is') 5.17
('card', 'payment') 4.0
('but', 'the') 3.17
('the', 'customer') 3.17
('the', 'card') 2.585
('the', 'password') 2.585

min_freq = 3
('but', 'the') 3.17
('the', 'card') 2.585


## Experiment 2 — Compare Association Measures

Different measures reward different statistical properties.

We compare:

- PMI,
- likelihood ratio,
- chi-square,
- Dice.

In [47]:
measures = {
    "PMI":
        BigramAssocMeasures.pmi,

    "Likelihood Ratio":
        BigramAssocMeasures.likelihood_ratio,

    "Chi-square":
        BigramAssocMeasures.chi_sq,

    "Dice":
        BigramAssocMeasures.dice,
}

for name, measure in measures.items():

    finder = BigramCollocationFinder.from_words(
        tokens
    )

    finder.apply_freq_filter(2)

    scores = finder.score_ngrams(
        measure
    )

    print(
        "\n",
        name
    )

    for pair, score in scores[:8]:

        print(
            pair,
            round(score, 3)
        )


 PMI
('account', 'is') 5.17
('card', 'payment') 4.0
('but', 'the') 3.17
('the', 'customer') 3.17
('the', 'card') 2.585
('the', 'password') 2.585

 Likelihood Ratio
('account', 'is') 15.534
('but', 'the') 13.722
('the', 'card') 11.284
('card', 'payment') 9.679
('the', 'customer') 9.022
('the', 'password') 5.411

 Chi-square
('account', 'is') 70.986
('card', 'payment') 29.972
('but', 'the') 24.511
('the', 'card') 19.565
('the', 'customer') 16.225
('the', 'password') 9.574

 Dice
('account', 'is') 0.667
('card', 'payment') 0.444
('the', 'card') 0.364
('but', 'the') 0.316
('the', 'customer') 0.222
('the', 'password') 0.211


## Collocation Tuning Recipe

For larger corpora:

1. normalize punctuation,
2. optionally filter stopwords,
3. set minimum frequency,
4. compare PMI and likelihood ratio,
5. inspect candidates manually,
6. adjust threshold,
7. consider domain-specific filters.

# Part XX — Statistical N-Gram Language Models

A language model assigns probability to sequences.

A bigram model approximates:

\[
P(w_t \mid w_{t-1})
\]

A trigram model approximates:

\[
P(w_t \mid w_{t-2}, w_{t-1})
\]

Increasing N gives more context but also increases sparsity.

In [48]:
tokenized_sentences = [
    [
        token.lower()
        for token in word_tokenize(
            doc["text"]
        )
        if token.isalpha()
    ]
    for doc in documents
]

## Maximum Likelihood Estimation

MLE estimates probabilities from observed relative frequencies.

Problem:

> unseen events receive probability 0.

In [49]:
train_data, vocab = padded_everygram_pipeline(
    2,
    tokenized_sentences,
)

mle_bigram = MLE(2)

mle_bigram.fit(
    train_data,
    vocab,
)

queries = [
    (
        "payment",
        ["card"],
    ),
    (
        "transaction",
        ["card"],
    ),
    (
        "banana",
        ["card"],
    ),
]

for word, context in queries:

    print(
        f"P({word} | {context}) =",
        mle_bigram.score(
            word,
            context,
        )
    )

P(payment | ['card']) = 0.3333333333333333
P(transaction | ['card']) = 0.16666666666666666
P(banana | ['card']) = 0.0


## Experiment — N-Gram Order

Compare unigram, bigram, and trigram models.

Expected trade-off:

```text
higher order
→ richer context
→ more sparsity
→ more unseen contexts
```

In [50]:
for order in [
    1,
    2,
    3,
]:

    train_data, vocab = padded_everygram_pipeline(
        order,
        tokenized_sentences,
    )

    model = MLE(
        order
    )

    model.fit(
        train_data,
        vocab,
    )

    if order == 1:

        score = model.score(
            "payment"
        )

        context = []

    elif order == 2:

        score = model.score(
            "payment",
            ["card"],
        )

        context = [
            "card"
        ]

    else:

        score = model.score(
            "payment",
            [
                "the",
                "card",
            ],
        )

        context = [
            "the",
            "card",
        ]

    print(
        "order =",
        order,
        "| context =",
        context,
        "| score =",
        score,
    )

order = 1 | context = [] | score = 0.020833333333333332
order = 2 | context = ['card'] | score = 0.3333333333333333
order = 3 | context = ['the', 'card'] | score = 0.5


# Part XXI — Language Model Smoothing

Smoothing redistributes probability mass so unseen events do not necessarily receive zero probability.

We compare:

- MLE
- Laplace
- Lidstone
- Witten-Bell
- Kneser-Ney

In [51]:
def train_lm(
    model_cls,
    order,
    **kwargs,
):

    train_data, vocab = padded_everygram_pipeline(
        order,
        tokenized_sentences,
    )

    model = model_cls(
        order,
        **kwargs,
    )

    model.fit(
        train_data,
        vocab,
    )

    return model

In [52]:
models = {
    "MLE":
        train_lm(
            MLE,
            2,
        ),

    "Laplace":
        train_lm(
            Laplace,
            2,
        ),

    "Lidstone_0.1":
        train_lm(
            Lidstone,
            2,
            gamma=0.1,
        ),

    "WittenBell":
        train_lm(
            WittenBellInterpolated,
            2,
        ),

    "KneserNey":
        train_lm(
            KneserNeyInterpolated,
            2,
        ),
}

for name, model in models.items():

    seen = model.score(
        "payment",
        ["card"],
    )

    unseen = model.score(
        "banana",
        ["card"],
    )

    print(
        f"{name:15s}",
        "seen =",
        round(seen, 6),
        "unseen =",
        round(unseen, 6),
    )

TypeError: Lidstone.__init__() got multiple values for argument 'gamma'

## Experiment — Lidstone Gamma

Lidstone smoothing adds a pseudo-count \(\gamma\).

Smaller gamma:

- closer to MLE.

Larger gamma:

- stronger smoothing.

In [ ]:
for gamma in [
    0.01,
    0.05,
    0.1,
    0.5,
    1.0,
]:

    model = train_lm(
        Lidstone,
        2,
        gamma=gamma,
    )

    print(
        "gamma =",
        gamma,
        "| seen =",
        round(
            model.score(
                "payment",
                ["card"],
            ),
            6,
        ),
        "| unseen =",
        round(
            model.score(
                "banana",
                ["card"],
            ),
            6,
        ),
    )

## Classical Text Generation

NLTK language models can sample words from learned distributions.

In [ ]:
kn_model = train_lm(
    KneserNeyInterpolated,
    3,
)

for seed in [
    1,
    7,
    42,
]:

    generated = kn_model.generate(
        num_words=12,
        random_seed=seed,
    )

    print(
        seed,
        "=>",
        " ".join(generated),
    )

# Part XXII — Corpus Linguistics

NLTK is particularly strong for corpus exploration.

It includes:

- corpus readers,
- concordance,
- frequency statistics,
- context analysis,
- tagged corpora,
- parsed corpora.

## Concordance

Frequency tells you how often a word occurs.

Concordance shows **how the word appears in context**.

In [ ]:
support_text = Text(
    tokens
)

support_text.concordance(
    "card",
    width=90,
    lines=10,
)

## Similar Words / Similar Contexts

Distributional intuition:

> words used in similar contexts may have related functions.

Small corpora provide weak evidence, so we also use the Brown corpus.

In [ ]:
brown_words = [
    word.lower()
    for word in brown.words()
    if word.isalpha()
]

brown_text = Text(
    brown_words
)

brown_text.similar(
    "money"
)

## Lexical Diversity

A simple type-token ratio is:

\[
rac{	ext{unique tokens}}
{	ext{total tokens}}
\]

Be cautious: this metric tends to decrease as corpus size increases.

In [ ]:
def lexical_diversity(tokens):

    if not tokens:
        return 0.0

    return (
        len(set(tokens))
        /
        len(tokens)
    )


print(
    "Case study:",
    lexical_diversity(tokens)
)

news_words = [
    word.lower()
    for word in brown.words(
        categories="news"
    )
    if word.isalpha()
]

print(
    "Brown news:",
    lexical_diversity(news_words)
)

## Built-In Corpora

In [ ]:
print(
    "Brown categories:"
)

print(
    brown.categories()
)

print(
    "\nReuters categories:"
)

print(
    reuters.categories()[:30]
)

print(
    "\nGutenberg files:"
)

print(
    gutenberg.fileids()
)

# Part XXIII — VADER Sentiment Analysis

VADER is a lexicon- and rule-based sentiment analyzer.

It handles several surface cues:

- negation,
- intensifiers,
- capitalization,
- punctuation.

In [ ]:
sia = SentimentIntensityAnalyzer()

tests = [
    "good",
    "very good",
    "VERY good",
    "VERY good!!!",
    "not good",
    "not very good",
    "really terrible",
]

for text in tests:

    print(
        f"{text:20s}",
        sia.polarity_scores(text)
    )

## Experiment — Compound Score

The compound score summarizes sentiment on a roughly \([-1,1]\) scale.

In [ ]:
for text in tests:

    score = sia.polarity_scores(
        text
    )["compound"]

    print(
        f"{text:20s}",
        score
    )

# Part XXIV — Edit Distance

Edit distance measures the number of edits required to transform one sequence into another.

Typical edits:

- insertion,
- deletion,
- substitution.

Applications:

- spelling correction,
- OCR cleanup,
- fuzzy matching,
- entity normalization.

In [ ]:
from nltk.metrics.distance import edit_distance

pairs = [
    (
        "transaction",
        "transation",
    ),
    (
        "account",
        "acount",
    ),
    (
        "payment",
        "paymant",
    ),
    (
        "fraud",
        "fraud",
    ),
]

for a, b in pairs:

    print(
        f"{a:12s}",
        f"{b:12s}",
        "distance =",
        edit_distance(
            a,
            b,
        ),
    )

## Experiment — Transpositions

In [ ]:
print(
    "standard:",
    edit_distance(
        "form",
        "from",
    )
)

print(
    "transposition-aware:",
    edit_distance(
        "form",
        "from",
        transpositions=True,
    )
)

# Part XXV — Classical Text Classification

NLTK contains classical classifiers such as Naive Bayes.

The key pipeline is:

```text
text
 ↓
feature extractor
 ↓
feature dictionary
 ↓
classifier
```

In classical NLP, feature engineering can strongly influence model performance.

In [ ]:
from nltk.classify import NaiveBayesClassifier

keywords = [
    "card",
    "payment",
    "account",
    "password",
    "transaction",
    "blocked",
    "login",
    "fraud",
]

def basic_features(text):

    words = {
        token.lower()
        for token in word_tokenize(text)
        if token.isalpha()
    }

    return {
        f"contains({keyword})":
            keyword in words
        for keyword in keywords
    }

In [ ]:
training_set = [
    (
        basic_features(
            doc["text"]
        ),
        doc["label"],
    )
    for doc in documents
]

classifier = NaiveBayesClassifier.train(
    training_set
)

classifier.show_most_informative_features(
    12
)

## Baseline Predictions

In [ ]:
test_texts = [
    "My credit card has an unauthorized purchase",
    "I cannot log in after password reset",
    "The merchant did not receive my payment",
]

for text in test_texts:

    print(
        text,
        "=>",
        classifier.classify(
            basic_features(text)
        ),
    )

# Part XXVI — Feature Engineering Experiment

We enrich the feature representation with:

- selected bigrams,
- negation,
- document length.

The classifier stays the same.

This lets us isolate the effect of **representation**.

In [ ]:
def richer_features(text):

    words = [
        token.lower()
        for token in word_tokenize(text)
        if token.isalpha()
    ]

    word_set = set(words)

    bigram_set = set(
        bigrams(words)
    )

    features = {
        f"contains({keyword})":
            keyword in word_set
        for keyword in keywords
    }

    selected_bigrams = [
        (
            "credit",
            "card",
        ),
        (
            "card",
            "payment",
        ),
        (
            "bank",
            "account",
        ),
        (
            "password",
            "reset",
        ),
        (
            "unauthorized",
            "transaction",
        ),
    ]

    for bg in selected_bigrams:

        features[
            f"bigram({bg[0]}_{bg[1]})"
        ] = bg in bigram_set

    features[
        "contains_negation"
    ] = any(
        word in {
            "not",
            "no",
            "never",
        }
        for word in words
    )

    features[
        "length_gt_10"
    ] = len(words) > 10

    return features

## Controlled Train/Test Comparison

The corpus is deliberately small, so accuracy is illustrative rather than statistically reliable.

The purpose is to observe model behaviour.

In [ ]:
train_docs = documents[:9]
test_docs = documents[9:]

def evaluate_feature_set(
    feature_fn
):

    train_set = [
        (
            feature_fn(
                doc["text"]
            ),
            doc["label"],
        )
        for doc in train_docs
    ]

    model = NaiveBayesClassifier.train(
        train_set
    )

    rows = []

    for doc in test_docs:

        prediction = model.classify(
            feature_fn(
                doc["text"]
            )
        )

        rows.append(
            (
                doc["label"],
                prediction,
                doc["text"],
            )
        )

    accuracy = (
        sum(
            gold == prediction
            for gold, prediction, text
            in rows
        )
        /
        len(rows)
    )

    return (
        accuracy,
        rows,
    )


for name, feature_fn in [
    (
        "basic",
        basic_features,
    ),
    (
        "richer",
        richer_features,
    ),
]:

    accuracy, rows = evaluate_feature_set(
        feature_fn
    )

    print(
        "\n",
        name,
        "accuracy =",
        accuracy,
    )

    for gold, prediction, text in rows:

        print(
            "gold =",
            gold,
            "| pred =",
            prediction,
            "|",
            text,
        )

## Production Perspective

For large text-classification problems, a stronger classical stack is usually:

```text
NLTK preprocessing / analysis
        ↓
TfidfVectorizer
        ↓
sparse matrix
        ↓
LogisticRegression / LinearSVC / SGDClassifier
```

NLTK remains useful for linguistic analysis even when scikit-learn performs the modelling.

# Part XXVII — Classical Sequence Taggers and Backoff

NLTK includes:

- `DefaultTagger`
- `RegexpTagger`
- `UnigramTagger`
- `BigramTagger`
- `TrigramTagger`
- Brill taggers
- HMM taggers

A useful classical idea is **backoff**:

```text
trigram tagger
   ↓ unknown
bigram tagger
   ↓
unigram tagger
   ↓
default tagger
```

In [ ]:
from nltk.tag import (
    DefaultTagger,
    UnigramTagger,
    BigramTagger,
    TrigramTagger,
)

tagged_sentences = treebank.tagged_sents()

split = int(
    len(tagged_sentences)
    * 0.8
)

train_sentences = tagged_sentences[:split]

test_sentences = tagged_sentences[split:]

default_tagger = DefaultTagger(
    "NN"
)

unigram_tagger = UnigramTagger(
    train_sentences,
    backoff=default_tagger,
)

bigram_tagger = BigramTagger(
    train_sentences,
    backoff=unigram_tagger,
)

trigram_tagger = TrigramTagger(
    train_sentences,
    backoff=bigram_tagger,
)

for name, tagger in [
    (
        "default",
        default_tagger,
    ),
    (
        "unigram",
        unigram_tagger,
    ),
    (
        "bigram",
        bigram_tagger,
    ),
    (
        "trigram",
        trigram_tagger,
    ),
]:

    print(
        f"{name:10s}",
        round(
            tagger.accuracy(
                test_sentences
            ),
            4,
        ),
    )

## Interpretation

Increasing context does not automatically guarantee improvement.

Higher-order taggers:

- memorize more context,
- encounter more unseen contexts,
- depend increasingly on backoff.

This mirrors the sparsity problem in N-gram language models.

# Part XXVIII — Classification Evaluation

Important metrics:

## Precision

Of everything predicted as class \(C\), how much was actually class \(C\)?

## Recall

Of everything that truly belongs to class \(C\), how much did the model recover?

## F1

Harmonic mean of precision and recall.

In [ ]:
from nltk.metrics import (
    precision,
    recall,
    f_measure,
    ConfusionMatrix,
)

gold = [
    "fraud",
    "account",
    "payment",
    "fraud",
    "payment",
    "fraud",
]

pred = [
    "fraud",
    "payment",
    "payment",
    "fraud",
    "account",
    "fraud",
]

print(
    ConfusionMatrix(
        gold,
        pred,
    )
)

for label in sorted(
    set(gold)
    |
    set(pred)
):

    gold_set = {
        i
        for i, y
        in enumerate(gold)
        if y == label
    }

    pred_set = {
        i
        for i, y
        in enumerate(pred)
        if y == label
    }

    print(
        label,
        "precision =",
        precision(
            gold_set,
            pred_set,
        ),
        "recall =",
        recall(
            gold_set,
            pred_set,
        ),
        "f1 =",
        f_measure(
            gold_set,
            pred_set,
        ),
    )

# Part XXIX — BLEU and METEOR

NLTK also provides metrics for machine translation and generated text.

## BLEU

BLEU is based largely on N-gram overlap.

It tends to reward lexical and phrase overlap.

In [ ]:
from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction,
)

reference = [
    [
        "the",
        "card",
        "payment",
        "was",
        "declined",
    ]
]

candidates = [
    "the card payment was declined".split(),
    "the card transaction was declined".split(),
    "payment failed".split(),
]

for candidate in candidates:

    score = sentence_bleu(
        reference,
        candidate,
        smoothing_function=
            SmoothingFunction().method1,
    )

    print(
        candidate,
        "=>",
        round(
            score,
            4,
        ),
    )

## Experiment — BLEU N-Gram Weights

In [ ]:
candidate = (
    "the card transaction was declined"
    .split()
)

weight_sets = {
    "unigram_only":
        (
            1.0,
            0,
            0,
            0,
        ),

    "unigram_bigram":
        (
            0.5,
            0.5,
            0,
            0,
        ),

    "BLEU4_default":
        (
            0.25,
            0.25,
            0.25,
            0.25,
        ),
}

for name, weights in weight_sets.items():

    score = sentence_bleu(
        reference,
        candidate,
        weights=weights,
        smoothing_function=
            SmoothingFunction().method1,
    )

    print(
        name,
        "=>",
        round(
            score,
            4,
        ),
    )

## METEOR

METEOR uses more flexible lexical matching than pure N-gram precision.

In [ ]:
from nltk.translate.meteor_score import meteor_score

reference = [
    [
        "the",
        "payment",
        "was",
        "completed",
        "successfully",
    ]
]

candidates = [
    "the payment was completed successfully".split(),
    "the transaction completed successfully".split(),
    "payment succeeded".split(),
]

for candidate in candidates:

    try:
        score = meteor_score(
            reference,
            candidate,
        )

        print(
            candidate,
            "=>",
            score,
        )

    except Exception as exc:

        print(
            candidate,
            "=>",
            exc,
        )

# Part XXX — Additional NLTK Capabilities

Your original list already covered most of NLTK's major NLP areas.

Important additions include:

| Capability | NLTK functionality |
|---|---|
| POS tagging | pretrained and trainable taggers |
| Sentence segmentation | Punkt |
| Tokenizer families | regex, Treebank, Tweet |
| Multi-word expressions | `MWETokenizer` |
| Conditional frequencies | `ConditionalFreqDist` |
| Concordance | `Text.concordance()` |
| Similar/common contexts | corpus-based lexical exploration |
| Lexical diversity | vocabulary analysis |
| VADER sentiment | rule-based sentiment |
| Sequence taggers | default, unigram, bigram, trigram, HMM, Brill |
| Edit distance | fuzzy string similarity |
| Parse-tree manipulation | `Tree` APIs |
| Probability distributions | classical statistical NLP |
| LM smoothing | Laplace, Lidstone, Witten-Bell, Kneser-Ney |
| Classical generation | `nltk.lm.generate()` |
| BLEU / METEOR / chrF / GLEU / NIST | text evaluation |
| WordNet similarity | path, Wu-Palmer, LCH, information-content metrics |
| Association measures | PMI, chi-square, likelihood ratio, Dice, Jaccard |
| Corpus readers | raw, categorized, tagged, parsed corpora |

# Part XXXI — Independent Experiment Lab

Use these exercises to actively tune each technique.

## Experiment 1 — Tokenizers
Compare vocabulary size and output for:

- Treebank,
- regex,
- Tweet tokenizer.

Use text containing:

- URLs,
- hashtags,
- numbers,
- contractions,
- hyphenated words.

---

## Experiment 2 — Stopwords

Compare:

1. no stopword removal,
2. standard NLTK stopwords,
3. negation-preserving stopwords,
4. custom banking stopwords.

Evaluate whether classification changes.

---

## Experiment 3 — Stemming

Compare vocabulary size after:

- no stemming,
- Porter,
- Snowball,
- Lancaster.

Look for over-stemming.

---

## Experiment 4 — Lemmatization

Compare:

- lemmatization without POS,
- POS-aware lemmatization.

Test:

```text
running
better
went
studies
```

---

## Experiment 5 — POS Tagging

Create examples where words change grammatical roles:

```text
record
transfer
charge
report
```

Inspect contextual POS predictions.

---

## Experiment 6 — Chunk Grammar Tuning

Start with:

```text
NN+
```

Then gradually add:

```text
JJ*
DT?
NN+
```

Track:

```text
false positives
false negatives
```

---

## Experiment 7 — NER

Create ten domain sentences containing:

- people,
- companies,
- cities,
- financial products.

Document errors.

---

## Experiment 8 — CFG Coverage

Start from a small grammar.

Add rules until five additional sentences parse successfully.

Observe how grammar complexity grows.

---

## Experiment 9 — PCFG

Construct an ambiguous grammar and modify production probabilities.

Observe how Viterbi ranking changes.

---

## Experiment 10 — WordNet

Study ambiguous words:

```text
bank
charge
interest
account
```

Inspect all synsets before computing similarity.

---

## Experiment 11 — WSD

For `"bank"`, create increasingly informative contexts:

```text
bank
money bank
loan deposit bank
river water bank
river shore fishing bank
```

Observe Lesk stability.

---

## Experiment 12 — Frequency Analysis

Compare:

```text
raw token frequency
vs
content-word frequency
```

Then repeat on Brown or Reuters.

---

## Experiment 13 — Collocations

Sweep:

```text
min_freq = 1, 2, 3, 5
```

Compare:

- PMI,
- likelihood ratio,
- chi-square,
- Dice.

---

## Experiment 14 — Language Model Order

Train:

- unigram,
- bigram,
- trigram,
- 4-gram.

Compare:

- seen sequence probability,
- unseen contexts,
- generated text.

---

## Experiment 15 — Smoothing

Compare:

- MLE,
- Laplace,
- Lidstone,
- Witten-Bell,
- Kneser-Ney.

Focus on unseen sequences.

---

## Experiment 16 — Lidstone Hyperparameter

Sweep:

```text
gamma =
0.01
0.05
0.1
0.5
1.0
```

Observe probability redistribution.

---

## Experiment 17 — Classification Features

Progressively add:

1. keyword indicators,
2. unigrams,
3. bigrams,
4. negation,
5. sentiment,
6. entity count,
7. document length.

Measure performance after every addition.

---

## Experiment 18 — Sequence Taggers

Compare:

```text
DefaultTagger
UnigramTagger
BigramTagger
TrigramTagger
```

Then compare with and without backoff.

---

## Experiment 19 — Error Analysis

For each NLP component maintain a table:

| Input | Expected | Actual | Error type | Possible fix |
|---|---|---|---|---|

This is one of the best ways to develop practical NLP intuition.

# Part XXXII — Complete NLTK Mental Model

```text
NLTK
│
├── TEXT PROCESSING
│   ├── sentence segmentation
│   ├── tokenization
│   ├── regex tokenization
│   ├── Tweet tokenization
│   ├── multi-word expressions
│   ├── stopwords
│   ├── stemming
│   └── lemmatization
│
├── LINGUISTIC ANNOTATION
│   ├── POS tagging
│   ├── chunking
│   └── Named Entity Recognition
│
├── SYNTAX
│   ├── CFG
│   ├── constituency parsing
│   ├── parse trees
│   ├── PCFG
│   ├── probabilistic parsing
│   └── dependency structures
│
├── LEXICAL SEMANTICS
│   ├── WordNet
│   ├── synsets
│   ├── synonyms
│   ├── antonyms
│   ├── hypernyms
│   ├── hyponyms
│   ├── meronyms
│   ├── semantic similarity
│   └── word-sense disambiguation
│
├── STATISTICAL NLP
│   ├── FreqDist
│   ├── ConditionalFreqDist
│   ├── N-grams
│   ├── collocations
│   ├── PMI
│   ├── chi-square
│   ├── likelihood ratio
│   ├── n-gram language modelling
│   └── smoothing
│
├── CORPUS LINGUISTICS
│   ├── concordance
│   ├── similar contexts
│   ├── lexical diversity
│   ├── Brown
│   ├── Reuters
│   ├── Gutenberg
│   └── Treebank
│
├── CLASSICAL NLP MODELS
│   ├── Naive Bayes
│   ├── default tagger
│   ├── unigram tagger
│   ├── bigram tagger
│   ├── trigram tagger
│   ├── HMM taggers
│   └── Brill taggers
│
├── OTHER UTILITIES
│   ├── VADER sentiment
│   ├── edit distance
│   ├── probability distributions
│   └── text generation
│
└── EVALUATION
    ├── precision
    ├── recall
    ├── F1
    ├── confusion matrix
    ├── BLEU
    └── METEOR
```

# Final Takeaway

NLTK is best understood as a **computational linguistics laboratory**.

It exposes NLP internals that modern end-to-end neural systems often abstract away.

Learning these mechanisms builds intuition around:

```text
representation
context
ambiguity
syntax
semantics
probability
sparsity
generalization
feature engineering
evaluation
```

Those concepts remain important even when you later move to:

```text
spaCy
scikit-learn
PyTorch
Hugging Face
transformers
LLMs
RAG
```

Modern NLP changes the modelling machinery, but many of the underlying problems remain the same.